In [ ]:
import os
import sys
sys.path.append('utils')

from model import simulate_portfolio, SegmentTree, Loan, CONFIG, REH_LEVELS, PARAM_TABLE, RehypoAccount, pad_loc, build_signals, load_price_df
from log_model import simulate_portfolio_log

import pandas as pd
from pathlib import Path
from datetime import timedelta
import random, numpy as np
from dataclasses import dataclass, field
from typing import Dict, List

random.seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"])

# Préstamos convencionales

# creación del préstamo

In [2]:
log, df = simulate_portfolio()
df

,loan_id,principal_mxn,principal_usd,coin,rehypo_frac,status,margin_calls,duration_days,interest_usd,liquidation_fee_usd,liquidation_rec_usd,rehypo_gain_usd,cash_inflow_usd,net_profit_usd,shortfall_usd
0,1,1500000,78947.37,BTC,0.4,liquidated_full,2,488.68,17926.51,8414.37,96765.31,-31364.21,91741.98,-5023.33,0
1,2,1500000,78947.37,BTC,0.0,repaid_maturity,0,364.94,14208.28,0.00,0.00,0.00,14208.28,14208.28,0
2,3,200000,10526.32,BTC,0.0,repaid_maturity,1,361.91,1878.68,0.00,0.00,0.00,1878.68,1878.68,0
3,4,1250000,65789.47,BTC,0.4,repaid_maturity,1,365.00,11157.83,0.00,0.00,86230.40,97388.23,97388.23,0
4,5,1750000,92105.26,BTC,0.0,repaid_maturity,0,361.49,16419.43,0.00,0.00,0.00,16419.43,16419.43,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,2000000,105263.16,ETH,0.0,repaid_maturity,0,729.94,42102.06,0.00,0.00,0.00,42102.06,42102.06,0
9996,9997,1000000,52631.58,ETH,0.0,repaid_maturity,0,730.00,21052.57,0.00,0.00,0.00,21052.57,21052.57,0
9997,9998,2000000,105263.16,ETH,0.4,liquidated_full,1,24.81,1356.50,11202.49,128828.61,-4565.15,136822.45,7993.84,0
9998,9999,200000,10526.32,ETH,0.0,repaid_maturity,0,365.00,2105.25,0.00,0.00,0.00,2105.25,2105.25,0


In [7]:
results = df[['principal_mxn','principal_usd', 'coin', 'rehypo_frac', 'status', 'duration_days', 'interest_usd', 'liquidation_fee_usd', 'rehypo_gain_usd', 'cash_inflow_usd', 'net_profit_usd']]
results['rehypo_frac'] = results['rehypo_frac'] * 100 
# Agregar % a rehypo_frac
results['rehypo_frac'] = results['rehypo_frac'].astype(str) + '%'
results

/var/folders/5j/3kf31kkn749_1dxbkmc_d2_r0000gn/T/ipykernel_47287/2609158765.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results['rehypo_frac'] = results['rehypo_frac'] * 100
/var/folders/5j/3kf31kkn749_1dxbkmc_d2_r0000gn/T/ipykernel_47287/2609158765.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results['rehypo_frac'] = results['rehypo_frac'].astype(str) + '%'


,principal_mxn,principal_usd,coin,rehypo_frac,status,duration_days,interest_usd,liquidation_fee_usd,rehypo_gain_usd,cash_inflow_usd,net_profit_usd
0,1500000,78947.37,BTC,40.0%,liquidated_full,488.68,17926.51,8414.37,-31364.21,91741.98,-5023.33
1,1500000,78947.37,BTC,0.0%,repaid_maturity,364.94,14208.28,0.00,0.00,14208.28,14208.28
2,200000,10526.32,BTC,0.0%,repaid_maturity,361.91,1878.68,0.00,0.00,1878.68,1878.68
3,1250000,65789.47,BTC,40.0%,repaid_maturity,365.00,11157.83,0.00,86230.40,97388.23,97388.23
4,1750000,92105.26,BTC,0.0%,repaid_maturity,361.49,16419.43,0.00,0.00,16419.43,16419.43
...,...,...,...,...,...,...,...,...,...,...,...
9995,2000000,105263.16,ETH,0.0%,repaid_maturity,729.94,42102.06,0.00,0.00,42102.06,42102.06
9996,1000000,52631.58,ETH,0.0%,repaid_maturity,730.00,21052.57,0.00,0.00,21052.57,21052.57
9997,2000000,105263.16,ETH,40.0%,liquidated_full,24.81,1356.50,11202.49,-4565.15,136822.45,7993.84
9998,200000,10526.32,ETH,0.0%,repaid_maturity,365.00,2105.25,0.00,0.00,2105.25,2105.25


In [8]:
results.to_csv('results.csv', index=False)

In [2]:
aux = pd.read_csv('results.csv')
aux

,principal_mxn,principal_usd,coin,rehypo_frac,status,duration_days,interest_usd,liquidation_fee_usd,rehypo_gain_usd,cash_inflow_usd,net_profit_usd
0,1500000,78947.37,BTC,40.0%,liquidated_full,488.68,17926.51,8414.37,-31364.21,91741.98,-5023.33
1,1500000,78947.37,BTC,0.0%,repaid_maturity,364.94,14208.28,0.00,0.00,14208.28,14208.28
2,200000,10526.32,BTC,0.0%,repaid_maturity,361.91,1878.68,0.00,0.00,1878.68,1878.68
3,1250000,65789.47,BTC,40.0%,repaid_maturity,365.00,11157.83,0.00,86230.40,97388.23,97388.23
4,1750000,92105.26,BTC,0.0%,repaid_maturity,361.49,16419.43,0.00,0.00,16419.43,16419.43
...,...,...,...,...,...,...,...,...,...,...,...
9995,2000000,105263.16,ETH,0.0%,repaid_maturity,729.94,42102.06,0.00,0.00,42102.06,42102.06
9996,1000000,52631.58,ETH,0.0%,repaid_maturity,730.00,21052.57,0.00,0.00,21052.57,21052.57
9997,2000000,105263.16,ETH,40.0%,liquidated_full,24.81,1356.50,11202.49,-4565.15,136822.45,7993.84
9998,200000,10526.32,ETH,0.0%,repaid_maturity,365.00,2105.25,0.00,0.00,2105.25,2105.25


In [4]:
aux = aux.rename(columns={'net_profit_usd': 'brute_profit_usd'})
aux

,principal_mxn,principal_usd,coin,rehypo_frac,status,duration_days,interest_usd,liquidation_fee_usd,rehypo_gain_usd,cash_inflow_usd,brute_profit_usd
0,1500000,78947.37,BTC,40.0%,liquidated_full,488.68,17926.51,8414.37,-31364.21,91741.98,-5023.33
1,1500000,78947.37,BTC,0.0%,repaid_maturity,364.94,14208.28,0.00,0.00,14208.28,14208.28
2,200000,10526.32,BTC,0.0%,repaid_maturity,361.91,1878.68,0.00,0.00,1878.68,1878.68
3,1250000,65789.47,BTC,40.0%,repaid_maturity,365.00,11157.83,0.00,86230.40,97388.23,97388.23
4,1750000,92105.26,BTC,0.0%,repaid_maturity,361.49,16419.43,0.00,0.00,16419.43,16419.43
...,...,...,...,...,...,...,...,...,...,...,...
9995,2000000,105263.16,ETH,0.0%,repaid_maturity,729.94,42102.06,0.00,0.00,42102.06,42102.06
9996,1000000,52631.58,ETH,0.0%,repaid_maturity,730.00,21052.57,0.00,0.00,21052.57,21052.57
9997,2000000,105263.16,ETH,40.0%,liquidated_full,24.81,1356.50,11202.49,-4565.15,136822.45,7993.84
9998,200000,10526.32,ETH,0.0%,repaid_maturity,365.00,2105.25,0.00,0.00,2105.25,2105.25


# 200,000

In [ ]:
# EJEMPLO PARA PRÉSTAMOS DE 200,000 MXN CON COLATERAL BTC Y SIN REHIPOTECAR

# 1.1 SACAMOS EL PROMEDIO DE GANANCIA BRUTA EN USD PARA LOS PRÉSTAMOS DE 200,000 MXN DE LOS RESULTADOS DE LAS SIMULACIONES
ganancia_bruta_200mil = aux[(aux['principal_mxn'] == 200000) & (aux['coin'] == 'BTC') & (aux['rehypo_frac'] == '0.0%')].brute_profit_usd.mean()

# 1.2 SACAMOS EL PROMEDIO DE DÍAS DE DURACIÓN PARA LOS PRÉSTAMOS DE 200,000 MXN
duracion_promedio_200mil = aux[(aux['principal_mxn'] == 200000) & (aux['coin'] == 'BTC') & (aux['rehypo_frac'] == '0.0%')].duration_days.mean()

# 1.3 CONVERTIMOS LA DURACIÓN A MESES DE 30 DÍAS
meses_duracion = duracion_promedio_200mil / 30

# 2. CALCULAMOS LA GANANCIA NETA, DESPUES DE LOS COSTOS VARIABLES

# 2.1 CALCULAMOS EL COSTO DE CUSTODIA =
# VALOR DEL PRÉSTAMO EN USD (PROMEDIO) * 2 (YA QUE TENEMOS UN LTV DEL 50% Y SE DEBERÁ CUSTODIAR EL DOBLE DEL VALOR DEL PRÉSTAMO) * 0.0004 * DURACIÓN EN MESES
custodia = aux[aux['principal_mxn'] == 200000].principal_usd.mean() * 2 * 0.0004 * meses_duracion

# 2.2 CALCULAMOS EL OUTGOING FEE =
# VALORE DEL PRÉSTAMO EN USD * 2 * 0.0015 * 2 TRANSACCIONES PROMEDIO AL AÑO
outgoing_fee = aux[aux['principal_mxn'] == 200000].principal_usd.mean() * 2 * 0.0015 * 2

# 2.3 CALCULAMOS EL COSTO DE KYC =
# 1.7 (VERIFICACIÓN DE IDENTIDAD Y CRUCE DE LISTAS) + COSTO ADICIONAL DE 2.75 USD POR CLIENTE (GESTOR DE EXPEDIENTE Y MONITOREO)
kyc = 1.7 + 2.75

# 2.4 OBTENEMOS EL COSTO TOTAL DE LOS PRÉSTAMOS DE 200,000 MXN
costos_variables = custodia + outgoing_fee + kyc

# 3. CALCULAMOS LA GANANCIA NETA
net_profit = ganancia_bruta_200mil - costos_variables
print(f"Ganancia neta por préstamo de 200,000 MXN: {net_profit:.2f} USD")

# 4. CALCULAMOS EL BREAKEVEN SABIENDO EL COSTO DE 30,090 DE COMPLIF ANUAL
breakeven = 30090 / net_profit
print(f"Breakeven: {breakeven:.2f} préstamos de 200,000 MXN al año")


Ganancia neta por préstamo de 200,000 MXN: 2083.33 USD
Breakeven: 14.44 préstamos de 200,000 MXN al año


# PARA TODOS

In [20]:
aux.principal_usd.mean()

np.float64(58562.368224)

In [12]:
# EJEMPLO PARA PRÉSTAMOS DE 200,000 MXN CON COLATERAL BTC Y SIN REHIPOTECAR

# 1.1 SACAMOS EL PROMEDIO DE GANANCIA BRUTA EN USD PARA LOS PRÉSTAMOS SIN REHYPO
ganancia_bruta_sinRehypo = aux[(aux['rehypo_frac'] == '0.0%')].brute_profit_usd.mean()

# 1.2 SACAMOS EL PROMEDIO DE DÍAS DE DURACIÓN PARA LOS PRÉSTAMOS SIN REHYPO
duracion_promedio_sinRehypo = aux[(aux['rehypo_frac'] == '0.0%')].duration_days.mean()

# 1.3 CONVERTIMOS LA DURACIÓN A MESES DE 30 DÍAS
meses_duracion = duracion_promedio_sinRehypo / 30

# 2. CALCULAMOS LA GANANCIA NETA, DESPUES DE LOS COSTOS VARIABLES

# 2.1 CALCULAMOS EL COSTO DE CUSTODIA =
# VALOR DEL PRÉSTAMO EN USD (PROMEDIO) * 2 (YA QUE TENEMOS UN LTV DEL 50% Y SE DEBERÁ CUSTODIAR EL DOBLE DEL VALOR DEL PRÉSTAMO) * 0.0004 * DURACIÓN EN MESES
custodia = aux.principal_usd.mean() * 2 * 0.0004 * meses_duracion

# 2.2 CALCULAMOS EL OUTGOING FEE =
# VALORE DEL PRÉSTAMO EN USD * 2 * 0.0015 * 2 TRANSACCIONES PROMEDIO AL AÑO
outgoing_fee = aux.principal_usd.mean() * 2 * 0.0015 * 2

# 2.3 CALCULAMOS EL COSTO DE KYC =
# 1.7 (VERIFICACIÓN DE IDENTIDAD Y CRUCE DE LISTAS) + COSTO ADICIONAL DE 2.75 USD POR CLIENTE (GESTOR DE EXPEDIENTE Y MONITOREO)
kyc = 1.7 + 2.75

# 2.4 OBTENEMOS EL COSTO TOTAL DE LOS PRÉSTAMOS DE 200,000 MXN
costos_variables = custodia + outgoing_fee + kyc

# 3. CALCULAMOS LA GANANCIA NETA
net_profit = ganancia_bruta_200mil - costos_variables
print(f"Ganancia neta por préstamo SIN REHYPO: {net_profit:.2f} USD")

# 4. CALCULAMOS EL BREAKEVEN SABIENDO EL COSTO DE 30,090 DE COMPLIF ANUAL
breakeven = 30090 / net_profit
print(f"Breakeven: {breakeven:.2f} préstamos SIN REHYPO al año")


Ganancia neta por préstamo SIN REHYPO: 12089.49 USD
Breakeven: 2.49 préstamos SIN REHYPO al año


In [6]:
# VALOR DE CADA VARIABLE
print("Ganancia bruta promedio por préstamo de 200,000 MXN:", ganancia_bruta_200mil)
print("Duración promedio en meses para préstamos de 200,000 MXN:", meses_duracion)
print("Costo de custodia:", custodia)
print("Outgoing fee:", outgoing_fee)
print("Costo de KYC:", kyc)
print("Costos variables totales:", costos_variables)
print("Ganancia neta por préstamo de 200,000 MXN:", net_profit)
print("Breakeven:", breakeven)

Ganancia bruta promedio por préstamo de 200,000 MXN: 2253.47972027972
Duración promedio en meses para préstamos de 200,000 MXN: 12.176285547785547
Costo de custodia: 102.53718246989277
Outgoing fee: 63.15792
Costo de KYC: 4.45
Costos variables totales: 170.14510246989275
Ganancia neta por préstamo de 200,000 MXN: 2083.3346178098273
Breakeven: 14.443191095069059


In [7]:
outgoing_fee = aux[aux['principal_mxn'] == 200000].principal_usd.mean() * 2 * 0.0015
outgoing_fee

np.float64(31.57896)

## Análisis

In [5]:
df = aux.copy()

In [6]:
df['rehypo_gain_usd'].sum()

np.float64(15447828.420000002)

In [7]:
df['brute_profit_usd'].sum()

np.float64(122789102.4)

In [8]:
df.groupby("principal_usd").size()

principal_usd
10526.32     1238
26315.79     1260
39473.68     1301
52631.58     1273
65789.47     1217
78947.37     1281
92105.26     1209
105263.16    1221
dtype: int64

In [10]:
df.groupby("principal_usd")['brute_profit_usd'].sum().sort_index(ascending=False)

principal_usd
105263.16    28013056.29
92105.26     22258759.88
78947.37     21264883.79
65789.47     16958080.57
52631.58     13244422.16
39473.68     11335591.75
26315.79      6745071.46
10526.32      2969236.50
Name: brute_profit_usd, dtype: float64

# Log-Préstamos

## Res

In [2]:
_, df_log = simulate_portfolio_log()
df_log

,loan_id,principal_mxn,principal_usd,coin,rehypo_frac,status,margin_calls,duration_days,interest_usd,liquidation_fee_usd,liquidation_rec_usd,rehypo_gain_usd,cash_inflow_usd,net_profit_usd,shortfall_usd
0,1,1500000,78947.37,BTC,0.2,liquidated_full,2,493.35,18652.63,8091.01,93046.67,-16890.53,102899.78,9853.12,0.0
1,2,1500000,78947.37,BTC,0.0,repaid_maturity,0,364.94,14208.28,0.00,0.00,0.00,14208.28,14208.28,0.0
2,3,200000,10526.32,BTC,0.1,repaid_maturity,1,361.91,1851.54,0.00,0.00,-699.49,1152.05,1152.05,0.0
3,4,1250000,65789.47,BTC,0.4,liquidated_full,1,1.44,44.03,6933.81,79738.76,-3417.80,83298.79,3560.03,0.0
4,5,1000000,52631.58,BTC,0.0,liquidated_full,1,119.34,3097.57,5248.69,60359.97,0.00,68706.22,8346.26,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,2000000,105263.16,ETH,0.0,repaid_maturity,0,361.54,20853.24,0.00,0.00,0.00,20853.24,20853.24,0.0
96,97,750000,39473.68,ETH,0.2,liquidated_full,1,163.16,3437.29,3905.09,44908.49,-4593.92,47656.96,2748.46,0.0
97,98,2000000,105263.16,ETH,0.3,liquidated_full,1,354.43,19645.46,10857.00,124855.53,-12012.92,143345.07,18489.53,0.0
98,99,500000,26315.79,ETH,0.3,liquidated_full,1,459.94,6373.49,2732.80,31427.16,-2903.88,37629.57,6202.41,0.0


In [3]:
results_log = df_log[['principal_mxn', 'coin', 'rehypo_frac', 'status', 'duration_days', 'interest_usd', 'liquidation_fee_usd', 'rehypo_gain_usd', 'cash_inflow_usd', 'net_profit_usd']]
results_log['rehypo_frac'] = results_log['rehypo_frac'] * 100 
# Agregar % a rehypo_frac
results_log['rehypo_frac'] = results_log['rehypo_frac'].astype(str) + '%'
results_log

/var/folders/5j/3kf31kkn749_1dxbkmc_d2_r0000gn/T/ipykernel_96896/160177203.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_log['rehypo_frac'] = results_log['rehypo_frac'] * 100
/var/folders/5j/3kf31kkn749_1dxbkmc_d2_r0000gn/T/ipykernel_96896/160177203.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_log['rehypo_frac'] = results_log['rehypo_frac'].astype(str) + '%'


,principal_mxn,coin,rehypo_frac,status,duration_days,interest_usd,liquidation_fee_usd,rehypo_gain_usd,cash_inflow_usd,net_profit_usd
0,1500000,BTC,20.0%,liquidated_full,493.35,18652.63,8091.01,-16890.53,102899.78,9853.12
1,1500000,BTC,0.0%,repaid_maturity,364.94,14208.28,0.00,0.00,14208.28,14208.28
2,200000,BTC,10.0%,repaid_maturity,361.91,1851.54,0.00,-699.49,1152.05,1152.05
3,1250000,BTC,40.0%,liquidated_full,1.44,44.03,6933.81,-3417.80,83298.79,3560.03
4,1000000,BTC,0.0%,liquidated_full,119.34,3097.57,5248.69,0.00,68706.22,8346.26
...,...,...,...,...,...,...,...,...,...,...
95,2000000,ETH,0.0%,repaid_maturity,361.54,20853.24,0.00,0.00,20853.24,20853.24
96,750000,ETH,20.0%,liquidated_full,163.16,3437.29,3905.09,-4593.92,47656.96,2748.46
97,2000000,ETH,30.0%,liquidated_full,354.43,19645.46,10857.00,-12012.92,143345.07,18489.53
98,500000,ETH,30.0%,liquidated_full,459.94,6373.49,2732.80,-2903.88,37629.57,6202.41


# Comparacion

In [35]:
cols = ['principal_mxn', 'rehypo_frac', 'coin', "margin_calls", "interest_usd", "rehypo_gain_usd",
        "cash_inflow_usd", "net_profit_usd", "status"]

cols2 = ["margin_calls", "interest_usd", "rehypo_gain_usd",
        "cash_inflow_usd", "net_profit_usd"]

comp = (df[["loan_id"] + cols]
        .merge(df_log[["loan_id"] + cols],
               on=["loan_id", 'principal_mxn', 'rehypo_frac', 'coin'],
               suffixes=("_lin", "_log")))

for c in cols2:
    comp[f"delta_{c}"] = comp[f"{c}_log"] - comp[f"{c}_lin"]
    comp[f"delta_{c}_pct"] = comp[f"delta_{c}"] / comp[f"{c}_lin"].replace(0, np.nan)

comp

,loan_id,principal_mxn,rehypo_frac,coin,margin_calls_lin,interest_usd_lin,rehypo_gain_usd_lin,cash_inflow_usd_lin,net_profit_usd_lin,status_lin,...,delta_margin_calls,delta_margin_calls_pct,delta_interest_usd,delta_interest_usd_pct,delta_rehypo_gain_usd,delta_rehypo_gain_usd_pct,delta_cash_inflow_usd,delta_cash_inflow_usd_pct,delta_net_profit_usd,delta_net_profit_usd_pct
0,1,1500000,0.2,BTC,2,18652.63,-16890.53,102899.78,9853.12,liquidated_full,...,0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0
1,2,1500000,0.0,BTC,0,14208.28,0.00,14208.28,14208.28,repaid_maturity,...,0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
2,3,200000,0.1,BTC,1,1851.54,-699.49,1152.05,1152.05,repaid_maturity,...,0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0
3,4,1250000,0.4,BTC,1,44.03,-3417.80,83298.79,3560.03,liquidated_full,...,0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0
4,5,1000000,0.0,BTC,1,3097.57,0.00,68706.22,8346.26,liquidated_full,...,0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,2000000,0.0,ETH,0,20853.24,0.00,20853.24,20853.24,repaid_maturity,...,0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
96,97,750000,0.2,ETH,1,3437.29,-4593.92,47656.96,2748.46,liquidated_full,...,0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0
97,98,2000000,0.3,ETH,1,19645.46,-12012.92,143345.07,18489.53,liquidated_full,...,0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0
98,99,500000,0.3,ETH,1,6373.49,-2903.88,37629.57,6202.41,liquidated_full,...,0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0


In [38]:
pd.crosstab(comp["status_lin"], comp["status_log"], margins=True)


status_log,liquidated_full,repaid_maturity,All
status_lin,,,
liquidated_full,61,0,61
repaid_maturity,0,39,39
All,61,39,100


In [41]:
from scipy.stats import ks_2samp, skew, kurtosis
ks_stat, ks_p = ks_2samp(df["net_profit_usd"], df_log["net_profit_usd"])
ks_stat

np.float64(0.11)

In [44]:
avg = 0.5 * (comp["net_profit_usd_lin"] + comp["net_profit_usd_log"])
diff = comp["net_profit_usd_log"] - comp["net_profit_usd_lin"]

diff

0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
     ... 
95    0.0
96    0.0
97    0.0
98    0.0
99    0.0
Length: 100, dtype: float64

In [49]:
lin = df.groupby('principal_mxn').size().reset_index(name='Lineal')
log = df_log.groupby('principal_mxn').size().reset_index(name='Log')

res = lin.merge(log, on='principal_mxn', how='outer').fillna(0).astype(int)
res

,principal_mxn,Lineal,Log
0,200000,8,8
1,500000,15,15
2,750000,7,7
3,1000000,14,14
4,1250000,15,15
5,1500000,14,14
6,1750000,13,13
7,2000000,14,14


In [50]:
lin = df.groupby('principal_mxn').agg(Intereses_Lineal=('interest_usd', 'sum'),
                                      Rehipo_Lineal=('rehypo_gain_usd', 'sum'),
                                      Cash_Inflow_Lineal=('cash_inflow_usd', 'sum'),
                                      Net_Profit_Lineal=('net_profit_usd', 'sum')).reset_index()
log = df_log.groupby('principal_mxn').agg(Intereses_Log=('interest_usd', 'sum'),
                                          Rehipo_Log=('rehypo_gain_usd', 'sum'),
                                          Cash_Inflow_Log=('cash_inflow_usd', 'sum'),
                                          Net_Profit_Log=('net_profit_usd', 'sum')).reset_index()

res = lin.merge(log, on='principal_mxn', how='outer').fillna(0).astype(float)
res

,principal_mxn,Intereses_Lineal,Rehipo_Lineal,Cash_Inflow_Lineal,Net_Profit_Lineal,Intereses_Log,Rehipo_Log,Cash_Inflow_Log,Net_Profit_Log
0,200000.0,5429.35,1064.55,87620.00,12983.98,8871.80,12758.13,149530.37,31861.96
1,500000.0,53761.74,124600.53,485296.98,202917.05,59927.70,154854.10,567096.87,242967.01
2,750000.0,32307.93,17358.77,313195.75,70749.02,52089.53,80461.64,771471.52,183664.81
3,1000000.0,82084.19,111262.43,944778.32,253461.16,120098.75,314344.65,1659733.66,532466.64
4,1250000.0,137721.88,66195.47,882983.22,258242.64,137721.88,66195.47,882983.22,258242.64
5,1500000.0,135365.30,83910.51,1036548.52,284657.64,167247.16,209825.71,1698006.40,482747.57
6,1750000.0,132647.01,111904.99,1095030.08,312590.26,210802.51,233549.51,1864853.79,557992.18
7,2000000.0,217881.25,19658.11,1205580.81,314982.67,282116.74,36840.78,1562931.29,418475.43


In [ ]:
lin = (df.groupby(['principal_mxn', 'status'])
         .size()
         .reset_index(name='n_lin'))

log = (df_log.groupby(['principal_mxn', 'status'])
          .size()
          .reset_index(name='n_log'))

res = (lin.merge(log, on=['principal_mxn', 'status'], how='outer')
          .fillna(0))

# Convertimos solo las columnas numéricas
res[['n_lin', 'n_log']] = res[['n_lin', 'n_log']].astype(int)
res

modelo,principal_mxn,status,Lineal,Log,delta,delta_pct
0,200000,liquidated_full,6,6,0,0.0
1,200000,repaid_maturity,2,2,0,0.0
2,500000,liquidated_full,9,9,0,0.0
3,500000,repaid_maturity,6,6,0,0.0
4,750000,liquidated_full,5,5,0,0.0
5,750000,repaid_maturity,2,2,0,0.0
6,1000000,liquidated_full,11,11,0,0.0
7,1000000,repaid_maturity,3,3,0,0.0
8,1250000,liquidated_full,8,8,0,0.0
9,1250000,repaid_maturity,7,7,0,0.0


In [59]:
lin = df.groupby(['status']).agg(Net_Profit_Lineal=('net_profit_usd', 'sum'))
log = df_log.groupby(['status']).agg(Net_Profit_Log=('net_profit_usd', 'sum'))

res = lin.merge(log, on='status', how='outer').fillna(0).astype(float)
res

,Net_Profit_Lineal,Net_Profit_Log
status,,
liquidated_full,373725.33,1371559.15
repaid_maturity,1336859.09,1336859.09


In [70]:
lin = df['duration_days'].mean()
log = df_log['duration_days'].mean()
res = pd.DataFrame({'Duration_Lineal': [lin], 'Duration_Log': [log]})
res

,Duration_Lineal,Duration_Log
0,240.2783,314.5773


In [71]:
lin = df.groupby('rehypo_frac').net_profit_usd.sum().reset_index(name='Net_Profit_Lineal')
log = df_log.groupby('rehypo_frac').net_profit_usd.sum().reset_index(name='Net_Profit_Log')
res = lin.merge(log, on='rehypo_frac', how='outer').fillna(0).astype(float)
res

,rehypo_frac,Net_Profit_Lineal,Net_Profit_Log
0,0.0,371187.52,375972.94
1,0.1,233718.33,356872.83
2,0.2,393104.74,835812.83
3,0.3,348977.91,736113.76
4,0.4,363595.92,403645.88


In [76]:
lin = df.groupby('rehypo_frac').size().reset_index(name='n_lin')
log = df_log.groupby('rehypo_frac').size().reset_index(name='n_log')
res = lin.merge(log, on='rehypo_frac', how='outer').fillna(0).astype(int)
res 

,rehypo_frac,n_lin,n_log
0,0,27,27
1,0,19,19
2,0,22,22
3,0,17,17
4,0,15,15


In [78]:
log

,rehypo_frac,n_log
0,0.0,27
1,0.1,19
2,0.2,22
3,0.3,17
4,0.4,15


# Pruebas

In [2]:
aux = pd.read_csv('results.csv')
aux = aux.rename(columns={'net_profit_usd': 'brute_profit_usd'})

In [3]:
aux

,principal_mxn,principal_usd,coin,rehypo_frac,status,duration_days,interest_usd,liquidation_fee_usd,rehypo_gain_usd,cash_inflow_usd,brute_profit_usd
0,1500000,78947.37,BTC,40.0%,liquidated_full,488.68,17926.51,8414.37,-31364.21,91741.98,-5023.33
1,1500000,78947.37,BTC,0.0%,repaid_maturity,364.94,14208.28,0.00,0.00,14208.28,14208.28
2,200000,10526.32,BTC,0.0%,repaid_maturity,361.91,1878.68,0.00,0.00,1878.68,1878.68
3,1250000,65789.47,BTC,40.0%,repaid_maturity,365.00,11157.83,0.00,86230.40,97388.23,97388.23
4,1750000,92105.26,BTC,0.0%,repaid_maturity,361.49,16419.43,0.00,0.00,16419.43,16419.43
...,...,...,...,...,...,...,...,...,...,...,...
9995,2000000,105263.16,ETH,0.0%,repaid_maturity,729.94,42102.06,0.00,0.00,42102.06,42102.06
9996,1000000,52631.58,ETH,0.0%,repaid_maturity,730.00,21052.57,0.00,0.00,21052.57,21052.57
9997,2000000,105263.16,ETH,40.0%,liquidated_full,24.81,1356.50,11202.49,-4565.15,136822.45,7993.84
9998,200000,10526.32,ETH,0.0%,repaid_maturity,365.00,2105.25,0.00,0.00,2105.25,2105.25


In [12]:
aux.groupby(['rehypo_frac', 'coin']).size()

rehypo_frac  coin
0.0%         BTC     2535
             ETH     2461
40.0%        BTC     2465
             ETH     2539
dtype: int64

In [10]:
aux.groupby('principal_usd').size()

principal_usd
10526.32     1238
26315.79     1260
39473.68     1301
52631.58     1273
65789.47     1217
78947.37     1281
92105.26     1209
105263.16    1221
dtype: int64

In [11]:
aux.groupby('principal_mxn').size()

principal_mxn
200000     1238
500000     1260
750000     1301
1000000    1273
1250000    1217
1500000    1281
1750000    1209
2000000    1221
dtype: int64

In [23]:
ganancia_bruta_sinRehypo = aux[(aux['rehypo_frac'] == '0.0%')].brute_profit_usd.mean()

duracion_promedio_sinRehypo = aux[(aux['rehypo_frac'] == '0.0%')].duration_days.mean()

meses_duracion = duracion_promedio_sinRehypo / 30

prestamo_promedio = aux[(aux['rehypo_frac'] == '0.0%')].principal_usd.mean()

intereses_promedio = aux[(aux['rehypo_frac'] == '0.0%')].interest_usd.mean()

In [17]:
ganancia_bruta_sinRehypo

np.float64(12982.849601681344)

In [18]:
meses_duracion

np.float64(11.473512009607685)

In [22]:
prestamo_promedio

np.float64(58920.82066853484)

In [24]:
intereses_promedio

np.float64(10510.951261008808)

In [31]:
ganancia_bruta_sinRehypo_200mil = aux[(aux['rehypo_frac'] == '0.0%') & (aux['principal_mxn'] == 200000)].brute_profit_usd.mean()

duracion_promedio_sinRehypo_200mil = aux[(aux['rehypo_frac'] == '0.0%') & (aux['principal_mxn'] == 200000)].duration_days.mean()

meses_duracion_200mil = duracion_promedio_sinRehypo_200mil / 30

prestamo_promedio_200mil = aux[(aux['rehypo_frac'] == '0.0%') & (aux['principal_mxn'] == 200000)].principal_usd.mean()

intereses_promedio = aux[(aux['rehypo_frac'] == '0.0%') & (aux['principal_mxn'] == 200000)].interest_usd.mean()

In [29]:
ganancia_bruta_sinRehypo_200mil

np.float64(2367.069508196721)

In [32]:
meses_duracion_200mil

np.float64(11.854093989071039)

In [33]:
prestamo_promedio_200mil

np.float64(10526.32)

# PRESTAMO VS VENTA

In [2]:
import pandas as pd

def compare_loan_vs_sell(results_df: pd.DataFrame) -> pd.DataFrame:
    need = ["coin","status","principal_usd","interest_usd","shortfall_usd",
            "start_price","end_price","collateral_btc_init_total","rehypo_frac"]
    for c in need:
        if c not in results_df.columns:
            raise ValueError(f"Falta columna requerida: {c}")

    df = results_df.copy()

    # Baseline: vender al inicio para obtener el principal
    df["btc_sold_baseline"] = df["principal_usd"] / df["start_price"]

    # BTC vendidos con préstamo (visión cliente)
    def btc_sold_with_loan(r):
        status = str(r["status"])
        if status.startswith("repaid"):  # repaid_maturity / repaid_end_of_data
            return (r["principal_usd"] + r["interest_usd"]) / r["end_price"]
        # Liquidación: se pierde el colateral inicial; si hay shortfall, vende extra al final
        sold = r["collateral_btc_init_total"]
        if r.get("shortfall_usd", 0) > 1e-9:
            sold += r["shortfall_usd"] / r["end_price"]
        return sold

    df["btc_sold_loan"] = df.apply(btc_sold_with_loan, axis=1)
    df["delta_btc_saved"] = df["btc_sold_baseline"] - df["btc_sold_loan"]  # >0 conviene pedir

    # (opcional) aislar casos sin rehypo para medir el puro efecto “préstamo”
    if "rehypo_frac" in df.columns:
        df = df.query("rehypo_frac == 0.0").copy()

    return (df.groupby(["coin","status"], as_index=False)
              .agg(loans=("delta_btc_saved","size"),
                   mean_delta_btc=("delta_btc_saved","mean"),
                   p50_delta_btc=("delta_btc_saved","median"),
                   p10=("delta_btc_saved", lambda s: s.quantile(0.10)),
                   p90=("delta_btc_saved", lambda s: s.quantile(0.90)),
                   neg_share=("delta_btc_saved", lambda s: (s < 0).mean()))
              .sort_values(["coin","status"]))


In [3]:
logs, df = simulate_portfolio()

# Para BI
results_display = df[[
    'loan_id','principal_mxn','principal_usd','coin','rehypo_frac','status',
    'duration_days','interest_usd','liquidation_fee_usd','liquidation_rec_usd',
    'rehypo_gain_usd','cash_inflow_usd','net_profit_usd',
    'margin_calls','start_ts','end_ts'
]].copy()
results_display['rehypo_frac_pct'] = (results_display['rehypo_frac']*100).round(0).astype(int).astype(str) + '%'

# Para el comparador (¡no toques rehypo_frac!)
results_compare = df[[
    'coin','status','principal_usd','interest_usd','shortfall_usd',
    'start_price','end_price','collateral_btc_init_total','rehypo_frac'
]].copy()

# Correr comparador
tbl_compare = compare_loan_vs_sell(results_compare)
print(tbl_compare.sort_values(['coin','status']).head(12))

  coin           status  loans  mean_delta_btc  p50_delta_btc        p10  \
0  BTC  liquidated_full    931       -1.397402      -1.341508  -2.570709   
1  BTC  repaid_maturity   1604        0.859699       0.664061   0.034589   
2  ETH  liquidated_full   1141      -22.325811     -20.622368 -41.564462   
3  ETH  repaid_maturity   1320        7.168029       5.604672  -3.758335   

         p90  neg_share  
0  -0.351620   1.000000  
1   2.108655   0.079800  
2  -5.345395   1.000000  
3  20.284251   0.182576  
